In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
from scipy.stats import zscore
import time
# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")

print("Scaled features shape:", X.shape)


#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

Scaled features shape: (2930, 172)


In [4]:
# Compute z-scores
z_scores = np.abs((X - X.mean()) / X.std())

# Define threshold
threshold = 3
# Get row indices where ANY feature is an outlier
outlier_indices = np.where((z_scores > threshold).any(axis=1))[0]

# Remove them
X_out = X.drop(index=X.index[outlier_indices])
print("Original features shape:", X.shape)
print("After Outlier Removal:", X_out.shape)


Original features shape: (2930, 172)
After Outlier Removal: (2766, 172)


In [5]:
#Apply StandardScaler
scaler = StandardScaler()
X_so = scaler.fit_transform(X_out)
print("Scaled shape:", X_so.shape)

Scaled shape: (2766, 172)


In [6]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_sop = pca.fit_transform(X_so)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_sop.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (2766, 24)
Explained variance ratio sum: 0.9506263088576382


In [7]:
#K-Means on Scaled + PCA + outliersremoval Data
start_time = time.time()
kmean_pca_out = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    kmean_pca_out.append({"algorithm": "KMeans", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")   

Runtime: 4.200109958648682 seconds
K-Means runtime: 4.2001 seconds


In [8]:
#Gaussian Mixture (GMM)on Scaled + PCA + outliersremoval Data
start_time = time.time()
gmm_pca_out = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    gmm_pca_out.append({"algorithm": "GMM", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")    

Runtime: 15.32729458808899 seconds
GMM runtime: 15.3273 seconds


In [9]:
#Agglomerative Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
agg_pca_out = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    agg_pca_out.append({"algorithm": "Agglomerative", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   


Runtime: 2.3148353099823 seconds
Agglomerative runtime: 2.3148 seconds


In [10]:
#Spectral Clustering on Scaled + PCA + outliersremoval Data
start_time = time.time()
spec_pca_out = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_sop)
    sil, db, ch = compute_metrics(X_sop, labels)
    spec_pca_out.append({"algorithm": "Spectral", "preprocessing": "PCA+Outliers", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")  

Runtime: 4.943235158920288 seconds
Spectral runtime: 4.9432 seconds


In [11]:
#DBSCAN on Scaled + PCA + outliersremoval Data
start_time = time.time()
dbscan_pca_out = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_sop)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_sop[mask], labels[mask])
        dbscan_pca_out.append({"algorithm": "DBSCAN", "preprocessing": "PCA+Outliers", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds") 

Runtime: 0.0499577522277832 seconds
DBSCAN runtime: 0.0500 seconds


In [12]:
from sklearn.cluster import Birch
start_time = time.time()

birch_pca_out = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X_sop)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_sop, labels)
        birch_pca_out.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA+Outliers",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds") 

Runtime: 4.185696601867676 seconds
BIRCH runtime: 4.1857 seconds


In [13]:
from sklearn.cluster import OPTICS
start_time = time.time()

optics_pca_out = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_sop)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_sop, labels)
        optics_pca_out.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA+Outliers",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds") 


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


Runtime: 9.452683687210083 seconds
Optics runtime: 9.4527 seconds


In [ ]:
import csv


ames_results_out_pca = (kmean_pca_out + gmm_pca_out + agg_pca_out + spec_pca_out + dbscan_pca_out+birch_pca_out + optics_pca_out)
# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_outliers_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_out_pca)